In [25]:
import pandas as pd
import ast

In [26]:
dados = pd.read_excel('teste_bruto.xlsx')
print(dados.head())

   Unnamed: 0  external_id  open_id                                 token  \
0           0          NaN      293  78b9341c-3252-4c09-9733-9e6f17f284bb   
1           1          NaN      294  791e9ea1-fdf6-486d-b783-6a2c7a7f1a46   
2           2          NaN      297  ae1f25c0-0774-463d-a3b8-483caf095e61   
3           3          NaN      298  308c1cca-4792-47be-848f-72c701af5d33   
4           4          NaN      300  57bc67d1-07a9-4f37-b79b-27fcd784b90b   

    status                                           name  folder_path  \
0  pending  Contrato Prest Serviços W. C. S Malta.doc.pdf  /Contratos/   
1  pending      Contrato Prest Serviços Fabiana Elisa.pdf  /Contratos/   
2  pending   Contrato Prest Serviços M.C.S Rossi Ltda.pdf  /Contratos/   
3  pending        Contrato Prest Serviços DW Comercio.pdf  /Contratos/   
4  pending         Contrato Prest Serviços Lab Brasil.pdf  /Contratos/   

                                       original_file  \
0  https://zapsign.s3.amazonaws.com/

In [ ]:
dados['signers'] = dados['signers'].astype(str)
dados['signers'] = dados['signers'].apply(ast.literal_eval)
signers = dados.explode('signers').reset_index(drop = True)

relat = signers.loc[
    signers['signers'].apply(
        lambda x: isinstance(x, dict) and x['status'] in ['nao_abriu', 'abriu']
    )
]

nomes_desejados = {
    'LUCAS GUILHERME PEIXOTO': 'lucas',
    'DAVID DE CASTRO GOMES': 'david',
    'GABRIELLA RIBEIRO AUGUSTO DA SILVA': 'gabi',
    'JULIANA GUERRA ASNAL': 'ju'
}

for nome_completo, apelido in nomes_desejados.items():
    df_temp = relat.loc[relat['signers'].apply(
        lambda x: isinstance(x, dict) and x['nome'] == nome_completo
    )].copy()
    df_temp['status'] = df_temp['signers'].apply(lambda x: x['status'])
    df_temp['link'] = df_temp['signers'].apply(lambda x: x['link_para_assinar'])
    
    # 2º: AGORA selecionar as colunas desejadas
    df_temp = df_temp[['name', 'open_id', 'status', 'link']].copy()
    
    # Renomear
    df_temp = df_temp.rename(columns={
        'name': 'Nome do Contrato',
        'open_id': 'ID do contrato',
        'status': 'Status',
        'link': 'Link p/ Assinatura'
    })
    



    df_temp.to_excel(f'{apelido}_pendentes.xlsx', index=False)
    print(f"{apelido}: {len(df_temp)} pendentes salvos")

outros = relat.loc[~relat['signers'].apply(
    lambda x: isinstance(x, dict) and x['nome'] in nomes_desejados
    
)]
outros.to_excel('outros_sig_pendentes.xlsx')
print(f"Outros signatários: {len(df_temp)} pendentes salvos")

signers.to_excel('signers.xlsx')
relat.to_excel('relat.xlsx')

lucas: 336 pendentes salvos
david: 325 pendentes salvos
gabi: 432 pendentes salvos
ju: 430 pendentes salvos
Outros signatários: 430 pendentes salvos
